# QKIDS: Interactive Demo
## Step 3: Test Your Trained Model Interactively!

This notebook lets you:
- Load the trained quantum kernel model
- Select test samples from the dataset
- Get predictions with confidence scores
- Compare predictions with actual labels

**Perfect for demonstrating your research!**

In [5]:
import sys
sys.path.append('../src')

import numpy as np
import joblib
import pickle
from preprocessing import NSLKDDPreprocessor
from quantum_kernel import create_quantum_kernel
from utils import print_section, print_subsection

import warnings
warnings.filterwarnings('ignore')

## Load Trained Model

In [6]:
print_section("LOADING TRAINED QUANTUM MODEL")

# Load quantum model
model_package = joblib.load('../saved_models/best_quantum_model.pkl')

quantum_svm = model_package['quantum_svm']
quantum_kernel = model_package['quantum_kernel']
n_qubits = model_package['n_qubits']
selected_features = model_package['selected_features']
X_train = model_package['X_train']
y_train = model_package['y_train']
performance = model_package['performance']

print(f"✅ Model loaded successfully!")
print(f"\n   Qubits: {n_qubits}")
print(f"   Training samples: {len(X_train)}")
print(f"   F1-Score: {performance['f1_score']:.4f}")
print(f"   Accuracy: {performance['accuracy']:.4f}")


  LOADING TRAINED QUANTUM MODEL

✅ Model loaded successfully!

   Qubits: 8
   Training samples: 800
   F1-Score: 0.8051
   Accuracy: 0.7700


## Load Test Samples

In [7]:
print_section("LOADING TEST SAMPLES")

# Load preprocessed test data
preprocessor = NSLKDDPreprocessor(
    train_path='', test_path='',
    sample_size=1000
)
data = preprocessor.load_preprocessed_data(f'preprocessed_{n_qubits}qubits.pkl')

X_test = data['X_test']
y_test = data['y_test']

print(f"✅ Test data loaded: {len(X_test)} samples")
print(f"   Normal samples: {(y_test == 0).sum()}")
print(f"   Attack samples: {(y_test == 1).sum()}")


  LOADING TEST SAMPLES


📂 Loaded preprocessed data from data\preprocessed_8qubits.pkl
✅ Test data loaded: 200 samples
   Normal samples: 100
   Attack samples: 100


## Helper Functions

In [8]:
def predict_sample(sample_idx, X_test, y_test, quantum_kernel, quantum_svm, X_train):
    """
    Predict a single test sample
    """
    # Get sample
    x_sample = X_test[sample_idx]
    y_true = y_test[sample_idx]
    
    print(f"\n🔍 Analyzing Sample #{sample_idx}...")
    print("─" * 60)
    
    # Compute quantum kernel vector
    k_vector = quantum_kernel.compute_kernel_vector(x_sample, X_train)
    k_vector = k_vector.reshape(1, -1)
    
    # Predict
    y_pred = quantum_svm.predict(k_vector)[0]
    
    # Get decision function value (confidence)
    decision_value = quantum_svm.decision_function(k_vector)[0]
    
    # Convert to probability-like score
    # Distance from hyperplane → confidence
    confidence = min(abs(decision_value) * 20, 99.9)  # Scale to percentage
    
    # Display results
    print("\n✨ PREDICTION RESULTS")
    print("=" * 60)
    
    label_map = {0: 'NORMAL', 1: 'ATTACK'}
    pred_label = label_map[y_pred]
    true_label = label_map[y_true]
    
    print(f"\n   Sample ID: {sample_idx}")
    print(f"   Predicted Class: {pred_label}")
    print(f"   Confidence: {confidence:.1f}%")
    print(f"   Actual Class: {true_label}")
    
    if y_pred == y_true:
        print(f"   Status: ✅ CORRECT")
    else:
        print(f"   Status: ❌ INCORRECT")
    
    print("\n" + "=" * 60)
    
    return y_pred, y_true, confidence


def show_sample_menu(X_test, y_test, n_samples=20):
    """
    Display available test samples
    """
    print("\n" + "=" * 70)
    print("  🔬 QKIDS: Quantum Kernel Intrusion Detection System")
    print("=" * 70)
    
    print(f"\n📊 Available Test Samples (showing {n_samples} of {len(X_test)}):")
    print("─" * 70)
    
    label_map = {0: 'Normal', 1: 'Attack'}
    
    # Show diverse samples
    normal_indices = np.where(y_test == 0)[0][:n_samples//2]
    attack_indices = np.where(y_test == 1)[0][:n_samples//2]
    
    all_indices = np.concatenate([normal_indices, attack_indices])
    np.random.shuffle(all_indices)
    
    for i, idx in enumerate(all_indices[:n_samples], 1):
        label = label_map[y_test[idx]]
        icon = "🟢" if y_test[idx] == 0 else "🔴"
        print(f"   [{i:2d}] Sample ID: {idx:4d}  |  Actual: {icon} {label}")
    
    print("─" * 70)
    
    return all_indices[:n_samples]

## Interactive Demo



In [9]:
# Show available samples
available_samples = show_sample_menu(X_test, y_test, n_samples=20)

print("\n💡 How to use:")
print("   1. Choose a sample number (1-20) from the list above")
print("   2. Run the prediction cell below")
print("   3. See the quantum kernel prediction results!")


  🔬 QKIDS: Quantum Kernel Intrusion Detection System

📊 Available Test Samples (showing 20 of 200):
──────────────────────────────────────────────────────────────────────
   [ 1] Sample ID:    1  |  Actual: 🟢 Normal
   [ 2] Sample ID:    0  |  Actual: 🟢 Normal
   [ 3] Sample ID:    4  |  Actual: 🟢 Normal
   [ 4] Sample ID:  101  |  Actual: 🔴 Attack
   [ 5] Sample ID:    5  |  Actual: 🟢 Normal
   [ 6] Sample ID:  108  |  Actual: 🔴 Attack
   [ 7] Sample ID:  105  |  Actual: 🔴 Attack
   [ 8] Sample ID:    6  |  Actual: 🟢 Normal
   [ 9] Sample ID:  107  |  Actual: 🔴 Attack
   [10] Sample ID:  106  |  Actual: 🔴 Attack
   [11] Sample ID:    2  |  Actual: 🟢 Normal
   [12] Sample ID:    3  |  Actual: 🟢 Normal
   [13] Sample ID:    9  |  Actual: 🟢 Normal
   [14] Sample ID:  103  |  Actual: 🔴 Attack
   [15] Sample ID:    8  |  Actual: 🟢 Normal
   [16] Sample ID:  100  |  Actual: 🔴 Attack
   [17] Sample ID:    7  |  Actual: 🟢 Normal
   [18] Sample ID:  109  |  Actual: 🔴 Attack
   [19] Sample ID:

### Make a Prediction

**Change the `choice` variable below to test different samples!**

In [10]:
# ========================================
# CHANGE THIS NUMBER (1-20) TO TEST!
# ========================================
choice = 1  # Try changing this to 2, 3, 4, etc.

# Validate choice
if 1 <= choice <= len(available_samples):
    sample_idx = available_samples[choice - 1]
    
    # Make prediction
    y_pred, y_true, confidence = predict_sample(
        sample_idx, X_test, y_test,
        quantum_kernel, quantum_svm, X_train
    )
else:
    print(f"❌ Invalid choice! Please enter a number between 1 and {len(available_samples)}")


🔍 Analyzing Sample #1...
────────────────────────────────────────────────────────────

⚛️  Computing kernel vector for new sample...


   Computing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 800/800 [00:07<00:00, 101.28it/s]

   ✓ Kernel vector computed: (800,)

✨ PREDICTION RESULTS

   Sample ID: 1
   Predicted Class: NORMAL
   Confidence: 17.4%
   Actual Class: NORMAL
   Status: ✅ CORRECT



## Batch Testing

### Test multiple samples at once

In [11]:
print_section("BATCH TESTING (5 Random Samples)")

# Test 5 random samples
random_samples = np.random.choice(available_samples, 5, replace=False)

results = []

for sample_idx in random_samples:
    y_pred, y_true, confidence = predict_sample(
        sample_idx, X_test, y_test,
        quantum_kernel, quantum_svm, X_train
    )
    
    results.append({
        'sample_id': sample_idx,
        'predicted': y_pred,
        'actual': y_true,
        'correct': y_pred == y_true,
        'confidence': confidence
    })

# Summary
print("\n" + "=" * 70)
print("  BATCH TESTING SUMMARY")
print("=" * 70)

correct = sum([r['correct'] for r in results])
total = len(results)
accuracy = correct / total * 100

print(f"\n   Total Samples: {total}")
print(f"   Correct: {correct}")
print(f"   Accuracy: {accuracy:.1f}%")
print(f"   Average Confidence: {np.mean([r['confidence'] for r in results]):.1f}%")


  BATCH TESTING (5 Random Samples)


🔍 Analyzing Sample #9...
────────────────────────────────────────────────────────────

⚛️  Computing kernel vector for new sample...


   Computing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 800/800 [00:07<00:00, 101.58it/s]


   ✓ Kernel vector computed: (800,)

✨ PREDICTION RESULTS

   Sample ID: 9
   Predicted Class: NORMAL
   Confidence: 22.8%
   Actual Class: NORMAL
   Status: ✅ CORRECT


🔍 Analyzing Sample #108...
────────────────────────────────────────────────────────────

⚛️  Computing kernel vector for new sample...


   Computing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 800/800 [00:07<00:00, 100.50it/s]


   ✓ Kernel vector computed: (800,)

✨ PREDICTION RESULTS

   Sample ID: 108
   Predicted Class: ATTACK
   Confidence: 26.2%
   Actual Class: ATTACK
   Status: ✅ CORRECT


🔍 Analyzing Sample #109...
────────────────────────────────────────────────────────────

⚛️  Computing kernel vector for new sample...


...

   ✓ Kernel vector computed: (800,)

✨ PREDICTION RESULTS

   Sample ID: 109
   Predicted Class: ATTACK
   Confidence: 26.2%
   Actual Class: ATTACK
   Status: ✅ CORRECT


🔍 Analyzing Sample #4...
────────────────────────────────────────────────────────────

⚛️  Computing kernel vector for new sample...


   Computing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 800/800 [00:07<00:00, 102.45it/s]


   ✓ Kernel vector computed: (800,)

✨ PREDICTION RESULTS

   Sample ID: 4
   Predicted Class: NORMAL
   Confidence: 33.0%
   Actual Class: NORMAL
   Status: ✅ CORRECT


🔍 Analyzing Sample #7...
────────────────────────────────────────────────────────────

⚛️  Computing kernel vector for new sample...


   Computing: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 800/800 [00:07<00:00, 102.08it/s]

   ✓ Kernel vector computed: (800,)

✨ PREDICTION RESULTS

   Sample ID: 7
   Predicted Class: NORMAL
   Confidence: 20.0%
   Actual Class: NORMAL
   Status: ✅ CORRECT


  BATCH TESTING SUMMARY

   Total Samples: 5
   Correct: 5
   Accuracy: 100.0%
   Average Confidence: 25.6%


## Compare with Classical Baseline

In [12]:
print_section("QUANTUM vs CLASSICAL COMPARISON")

# Load classical model
classical_package = joblib.load('../saved_models/best_classical_model.pkl')
classical_model = classical_package['model']
classical_perf = classical_package['performance']

print("\n📊 Performance Comparison:")
print("─" * 70)
print(f"\n{'Metric':<20} {'Quantum Kernel':<20} {'Classical RBF':<20}")
print("─" * 70)

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
for metric in metrics:
    q_val = performance[metric]
    c_val = classical_perf[metric]
    winner = "🏆" if q_val > c_val else ""
    print(f"{metric.replace('_', ' ').title():<20} {q_val:<20.4f} {c_val:<20.4f} {winner}")

print("─" * 70)


  QUANTUM vs CLASSICAL COMPARISON


📊 Performance Comparison:
──────────────────────────────────────────────────────────────────────

Metric               Quantum Kernel       Classical RBF       
──────────────────────────────────────────────────────────────────────
Accuracy             0.7700               0.7650               🏆
Precision            0.6985               0.6934               🏆
Recall               0.9500               0.9500               
F1 Score             0.8051               0.8017               🏆
──────────────────────────────────────────────────────────────────────


## Conclusion

You've successfully:
- ✅ Trained quantum and classical models
- ✅ Tested individual samples interactively
- ✅ Compared quantum vs classical performance

**For your research paper:**
- Use the metrics from notebook 02 for your results section
- Include the qubit ablation plot
- Show the model comparison figures
- Use this demo to showcase your system in presentations!

**🎉 Great work on your QKIDS project!**